The goal of this notebook is to train some adversarial patches on the CIFAR10 data set and visualize what these patches look like, as well as explore the effectiveness of patches once we alter certain features.

This notebook is largely modelled after: https://github.com/AIPI-590-XAI/Duke-AI-XAI/blob/main/adversarial-ai-example-notebooks/adversarial_attacks_patches.ipynb

I am writing my own version of tihs notebook to build a stronger understanding of this work.

In [ ]:
# Install necessary packages if they do not exist.
# %pip install numpy matplotlib torch torchvision

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   -------------------- ------------------- 4.7/9.3 MB 28.5 MB/s eta 0:00:01
   ------------------------------ --------- 7.1/9.3 MB 31.4 MB/s eta 0:00:01
   ------------------------------- -------- 7.3/9.3 MB 14.3 MB/s eta 0:00:01
   --------------------------------- ------ 7.9/9.3 MB 11.0 MB/s eta 0:00:01
   ----------------------------------- ---- 8.4/9.3 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------  9.2/9.3 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 6.8 MB/s  0:00:01
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   -- ------------------------------------- 8.7/124.1 MB 43.0 MB/s eta 0:00:03
   ----- ---------------------------------- 16.8


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import numpy as np
import math


import torch

import torchvision
from torchvision.datasets import CIFAR10
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")


There exist many kinds of models that torchvision offers. In terms of image classification, ResNet50 is mainstream. Choosing default pre-trained weights for ResNet50 forces the most up-to-date version.

In [ ]:
pretrained_model = torchvision.models.resnet50(weights = ResNet50_Weights.DEFAULT)
pretrained_model.to(device) # moves model to device
pretrained_model.eval() # Move to eval mode. Disable batch normalization

for p in pretrained_model.parameters():
    p.requires_grad = False

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con